In [ ]:

import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import json


df = pd.read_excel(r"D:\SIGNAL_PROJECT\decisiontree\dataset.xlsx")


df.columns = df.columns.str.strip()


df["ROLL"] = df["ROLL"].astype(str).str.replace("°", "", regex=False).astype(float)
df["PITCH"] = df["PITCH"].astype(str).str.replace("°", "", regex=False).astype(float)


df["AX"] = pd.to_numeric(df["AX"], errors="coerce")
df["AY"] = pd.to_numeric(df["AY"], errors="coerce")
df["AZ"] = pd.to_numeric(df["AZ"], errors="coerce")


df = df.dropna(subset=["AX", "AY", "AZ", "ROLL", "PITCH", "STATUS"])


X = df[["AX", "AY", "AZ", "ROLL", "PITCH"]]
y = df["STATUS"]


le = LabelEncoder()
y_encoded = le.fit_transform(y)


X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42
)


model = tf.keras.Sequential([
    tf.keras.layers.Dense(16, activation='relu', input_shape=(X_train.shape[1],)),
    tf.keras.layers.Dense(8, activation='relu'),
    tf.keras.layers.Dense(len(le.classes_), activation='softmax')
])

model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])


history = model.fit(
    X_train, y_train,
    epochs=40,
    batch_size=8,
    validation_split=0.2,
    verbose=1
)


loss, acc = model.evaluate(X_test, y_test)
print(f"\n✅ TensorFlow Model Accuracy: {round(acc, 3)}")

model.save("posture_model_tf.h5")
print("✅ TensorFlow model saved as posture_model_tf.h5")



converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

with open("posture_model.tflite", "wb") as f:
    f.write(tflite_model)

print("✅ TFLite model saved as posture_model.tflite")


with open("posture_labels.json", "w") as f:
    json.dump(le.classes_.tolist(), f)

print("✅ Labels saved as posture_labels.json")


preds = model.predict(X_test[:5])
for i, p in enumerate(preds):
    pred_label = le.inverse_transform([p.argmax()])[0]
    print(f"Sample {i+1}: Predicted = {pred_label}")


Epoch 1/40


d:\SIGNAL_PROJECT\sigenv\Lib\site-packages\keras\src\layers\core\dense.py:92: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


45/45 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.5404 - loss: 4.3876 - val_accuracy: 0.5444 - val_loss: 2.5313
Epoch 2/40
45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.5850 - loss: 1.4716 - val_accuracy: 0.7111 - val_loss: 0.9427
Epoch 3/40
45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7577 - loss: 0.7362 - val_accuracy: 0.7556 - val_loss: 0.6691
Epoch 4/40
45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7827 - loss: 0.5527 - val_accuracy: 0.8111 - val_loss: 0.4708
Epoch 5/40
45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8301 - loss: 0.4388 - val_accuracy: 0.8333 - val_loss: 0.3954
Epoch 6/40
45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8106 - loss: 0.3983 - val_accuracy: 0.8778 - val_loss: 0.3566
Epoch 7/40
45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8552 - loss: 0.3538 - val_accuracy: 0.8889 - val_loss: 0.3208
Epoch 8/40
45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8663 - loss: 0.3207 - val_accuracy: 0.8667 - val_loss: 0.2916
Epo


✅ TensorFlow Model Accuracy: 0.982
✅ TensorFlow model saved as posture_model_tf.h5
INFO:tensorflow:Assets written to: C:\Users\RKIRTH~1\AppData\Local\Temp\tmpkie7e9_w\assets


INFO:tensorflow:Assets written to: C:\Users\RKIRTH~1\AppData\Local\Temp\tmpkie7e9_w\assets


Saved artifact at 'C:\Users\RKIRTH~1\AppData\Local\Temp\tmpkie7e9_w'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 5), dtype=tf.float32, name='keras_tensor')
Output Type:
  TensorSpec(shape=(None, 2), dtype=tf.float32, name=None)
Captures:
  1930764014352: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1930764017232: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1930764015888: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1930764017616: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1930764017424: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1930764018000: TensorSpec(shape=(), dtype=tf.resource, name=None)
✅ TFLite model saved as posture_model.tflite
✅ Labels saved as posture_labels.json
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step
Sample 1: Predicted = ABNORMAL
Sample 2: Predicted = ABNORMAL
Sample 3: Predicted = NORMAL
Sample 4: Predicted = NORMAL
Sample 5: Predicted = NORMAL
